# 2. Feature Engineering and Target Construction

Turns the cached raw data into the 64-event modelling table. Cheap to re-run, so this is
where feature changes are iterated.

Two things worth watching in the output below: the **stationarity gate**, which excludes
any feature carrying a unit root before a model ever sees it, and the **damage
provenance** count, which shows how much of the damage signal is real and how much is an
EM-DAT reporting artefact.

## 2.1 Environment and cached inputs

Loads `_shared.py` (paths, the artifact cache helpers, the target definitions) and applies the thesis figure style, then prints the artifact cache so it is visible which upstream stage produced these inputs and when. Reads `market`, `disasters`, `macro` and `sp500` from stage 01.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "_shared.py").exists() else Path.cwd() / "notebooks"))
from _shared import *  # noqa: F401,F403  paths, artifact cache, target bounds

from src.evaluation import figures as fx
fx.apply_thesis_style()

print("stage inputs available in the artifact cache:")
print(artifact_status().to_string(index=False) if len(artifact_status()) else "  (none yet)")

market_clean = load_frame("market")
disasters_raw = load_frame("disasters")
macro = load_frame("macro")
sp500 = load_frame("sp500")
print(f"\nloaded: market={len(market_clean)} rows, disasters={len(disasters_raw)}, "
      f"macro={len(macro)}, sp500={len(sp500)}")


stage inputs available in the artifact cache:
                     artifact               written_utc rows                                                                                                                                                                                  note
       ablation_block_contrib 2026-09-16T17:08:06+00:00   60                                                                                                                                 Paired-bootstrap contribution of each external block.
              ablation_blocks 2026-09-16T17:07:55+00:00   72                                                                                                                                             Pooled R2/RMSE per external-block config.
       classification_summary 2026-09-16T16:08:49+00:00   20                                                                                                                                     Pre-registered binary labels, po

## 2.2 Market features

Two real bugs, independent of the blueprint, were found and fixed in `feature_eng.py` during this review:

1. **Look-ahead in SMA/EMA.** The rolling averages were computed directly on the *unshifted* price series, so `sma_5` at row *t* could include the price at *t* itself — the disaster's own shock day. The thesis's "strict pre-shock boundary" (§3.5.1: *"all rolling features must stop at t‑1"*) was violated. **Fixed**: rolling windows now compute over `price.shift(1)`.
2. **Missing 30-day panic-proxy column.** The thesis (§3.5.2, and the ATV definition in §3.2.2) calls for a dedicated 30-day rolling volatility feature, separate from the 5/10/20-day momentum SMAs. It was absent. **Added**: `rolling_std_30`.

**Deviation flagged and resolved (PPP adjustment, thesis §3.5.3):** the disaster damage figures are already CPI-adjusted at the source — EM-DAT's own `"Total Damage, Adjusted ('000 US$)"` column (verified against doc.emdat.be to use OECD CPI relative to Start Year, §1.3 above) is used in place of a separately-coded World Bank deflator step. Where EM-DAT has no adjusted figure (common — see the `damage_source` breakdown above), the unadjusted raw figure is used as a documented fallback, never silently zero-filled without a trace.

In [2]:
from src.data_pipeline.feature_eng import FeatureEngineer

fe = FeatureEngineer()

market_feats = fe.engineer_market_features(market_clean)
print("Engineered market features:")
print([c for c in market_feats.columns if c not in ("date", "aspi_close", "trading_volume", "price_source", "volume_source")])


Engineered market features:
['log_return', 'lag_return_t-1', 'lag_return_t-2', 'lag_return_t-3', 'lag_return_t-5', 'sma_5', 'ema_5', 'price_to_sma_5', 'price_to_ema_5', 'sma_10', 'ema_10', 'price_to_sma_10', 'price_to_ema_10', 'sma_20', 'ema_20', 'price_to_sma_20', 'price_to_ema_20', 'rolling_std_5', 'rolling_std_10', 'rolling_std_20', 'rolling_std_30', 'squared_return', 'garch_cond_vol', 'vol_ratio_1_30', 'vol_ratio_5_30', 'vol_ratio_10_30', 'vol_cv_30', 'log_vol_change_1']


### 2.2.1 Stationarity check

Augmented Dickey-Fuller and KPSS on every engineered market column, not just on
`log_return`. The moving-average columns are rolling means of a price level that
climbs from about 574 to over 10,000 across the sample; under a chronological split a
non-stationary feature puts the test fold outside any range the model saw in training.

In [3]:
import warnings

from statsmodels.tsa.stattools import adfuller, kpss

# Stationarity verification (thesis \u00a73.5.1). An earlier version of this cell tested
# log_return ALONE and reported "stationary", while six engineered price-level columns
# (sma_5/10/20, ema_5/10/20) went untested -- they are rolling means of the raw ASPI
# index, which runs from ~574 in 2000 to over 10,000 by 2022. Under a chronological
# split that guarantees the test fold occupies a feature range the training fold never
# saw. Every engineered market feature is now tested, and the verdict decides which
# columns are admitted to the model (see the EXCLUDE_COLS block in \u00a77).
#
# Convention: ADF null = has a unit root (want p < 0.05); KPSS null = is stationary
# (want p > 0.05). Only the ADF result gates exclusion -- see the reasoning in
# stationarity_verdict below.
def stationarity_verdict(series: pd.Series):
    s = series.dropna()
    if len(s) < 30 or s.nunique() < 3:
        return None, None, "too few observations"
    adf_p = adfuller(s)[1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # KPSS warns when p is clipped at a table bound
        kpss_p = kpss(s, nlags="auto")[1]
    # The disqualifying condition for a CHRONOLOGICAL split is a UNIT ROOT: a series
    # whose level wanders without reverting, so a later test fold occupies a range the
    # training fold never saw. That is exactly what ADF tests, so failing to reject its
    # null (p >= 0.05) is the exclusion criterion.
    #
    # KPSS is reported but does NOT gate inclusion. The volatility columns
    # (rolling_std_*, squared_return) reject the ADF unit-root null decisively
    # (p ~ 1e-11 or smaller) while still failing KPSS level-stationarity -- the ordinary
    # signature of persistent, mean-reverting volatility clustering, not of a trending
    # level. Excluding those would discard the thesis's own mandated 30-day panic proxy
    # (Sec. 3.5.2) over a property that creates no extrapolation risk. An earlier version
    # of this cell required both tests to agree and dropped 11 of 22 features on that
    # mistake.
    if adf_p >= 0.05:
        verdict = "UNIT ROOT (excluded)"
    elif kpss_p > 0.05:
        verdict = "stationary"
    else:
        verdict = "persistent, mean-reverting (kept)"
    return adf_p, kpss_p, verdict


market_feature_cols = [
    c for c in market_feats.columns
    if c not in ("date", "aspi_close", "trading_volume", "price_source", "volume_source")
    and pd.api.types.is_numeric_dtype(market_feats[c])
]

stationarity_rows = []
for col in market_feature_cols:
    adf_p, kpss_p, verdict = stationarity_verdict(market_feats[col])
    stationarity_rows.append({"feature": col, "adf_p": adf_p, "kpss_p": kpss_p, "verdict": verdict})
stationarity_table = pd.DataFrame(stationarity_rows)

NON_STATIONARY_COLS = stationarity_table.loc[
    stationarity_table["verdict"] == "UNIT ROOT (excluded)", "feature"
].tolist()
print(f"{len(NON_STATIONARY_COLS)} of {len(market_feature_cols)} engineered market features "
      f"carry a unit root and are excluded from the model in \u00a77:")
for col in NON_STATIONARY_COLS:
    print(f"  - {col}")
print()
stationarity_table


6 of 28 engineered market features carry a unit root and are excluded from the model in §7:
  - sma_5
  - ema_5
  - sma_10
  - ema_10
  - sma_20
  - ema_20



,feature,adf_p,kpss_p,verdict
0,log_return,3.589917e-26,0.100000,stationary
1,lag_return_t-1,3.607647e-26,0.100000,stationary
2,lag_return_t-2,3.625771e-26,0.100000,stationary
3,lag_return_t-3,3.644526e-26,0.100000,stationary
4,lag_return_t-5,3.685945e-26,0.100000,stationary
5,sma_5,9.965644e-01,0.010000,UNIT ROOT (excluded)
6,ema_5,9.934591e-01,0.010000,UNIT ROOT (excluded)
7,price_to_sma_5,2.852176e-25,0.100000,stationary
8,price_to_ema_5,3.298104e-24,0.100000,stationary
9,sma_10,9.936346e-01,0.010000,UNIT ROOT (excluded)


## 2.3 Disaster features and the inclusion filter

Builds the severity, hazard-magnitude and disaster-type columns, and applies the
inclusion filter: biological events excluded, and `population_affected` at or above
the configured threshold.

In [4]:
disaster_feats = fe.engineer_disaster_features(disasters_raw)
print(f"After thesis inclusion filter (>=1000 affected, exclude Biological/epidemic events): {disaster_feats.shape[0]} qualifying disasters")
print(disaster_feats["disaster_type"].value_counts())
print(f"\nThesis's own claimed scope (\u00a73.5.4/\u00a77.3): N \u2248 50-86 qualifying events. "
      f"Actual filtered count ({disaster_feats.shape[0]}) falls inside that range -- real sanity check passed.")


After thesis inclusion filter (>=1000 affected, exclude Biological/epidemic events): 94 qualifying disasters
disaster_type
Flood      69
Storm      15
Drought     7
Other       3
Name: count, dtype: int64

Thesis's own claimed scope (§3.5.4/§7.3): N ≈ 50-86 qualifying events. Actual filtered count (94) falls inside that range -- real sanity check passed.


### 2.3.1 Threshold transparency check

**Transparency check, not a scope change:** the thesis mandates the `>=1000 affected` threshold (§3.3.2) — that stays the modeled scope, unchanged, everywhere in this notebook. This is only a read of how many additional real EM-DAT records exist at other thresholds, so the choice of 1000 is visibly a real constraint, not an unexamined one.

In [5]:
clean_type = disasters_raw["disaster_type"].astype(str).str.strip().str.lower()
non_bio = disasters_raw[~clean_type.isin({"epidemic", "biological", "biological disaster", "pandemic"})]

print("Non-biological EM-DAT records qualifying at each affected-threshold (current thesis threshold: 1000):")
for thresh in (100, 500, 1000, 5000, 10000):
    n = int((non_bio["population_affected"].fillna(0) > thresh).sum())
    marker = "  <-- current modeled scope" if thresh == 1000 else ""
    print(f"  > {thresh:>6,d} affected: {n:>3d} records{marker}")


Non-biological EM-DAT records qualifying at each affected-threshold (current thesis threshold: 1000):
  >    100 affected: 100 records
  >    500 affected:  96 records
  >  1,000 affected:  94 records  <-- current modeled scope
  >  5,000 affected:  89 records
  > 10,000 affected:  80 records


## 2.4 Scope: which events the market series can carry

An event is modelled only if the ASPI series covers its full 90-trading-day recovery
window. Stage 01 now extends that series past the old 2023-06-28 archive cutoff using
countryeconomy.com, so ten previously-excluded events enter the study -- including both
Storm Ditwah dates.

The inclusion filter itself is unchanged: the same pre-registered >=1000-affected,
natural-disaster, non-epidemic criteria apply. Only the market series got longer.

In [6]:
ARCHIVE_END = market_clean["date"].max()
in_scope = disaster_feats[disaster_feats["event_date"] <= ARCHIVE_END].copy()
out_of_scope = disaster_feats[disaster_feats["event_date"] > ARCHIVE_END].copy()

print(f"Archive real coverage ends: {ARCHIVE_END.date()}")
print(f"In-scope disasters (modeled):      {in_scope.shape[0]}")
print(f"Out-of-scope disasters (excluded): {out_of_scope.shape[0]}  <- includes Ditwah")
out_of_scope[["event_date", "disaster_type", "event_name"]]


Archive real coverage ends: 2026-09-11
In-scope disasters (modeled):      94
Out-of-scope disasters (excluded): 0  <- includes Ditwah


,event_date,disaster_type,event_name


## 2.5 Target construction

**A correction made during this notebook's own development, shown here rather than hidden:** an earlier pass flagged Y1 as needing to be replaced with a market-model abnormal return (α + β·R_market, via the AR/CAR formulas in thesis §2.2.3, citing Fama et al. 1969 and MacKinlay 1997). On re-reading, that was a misattribution — **§2.2.3 is literature-review background** on classical event-study theory, presented there specifically because the thesis goes on to critique its "structural weaknesses" as motivation for using ML instead of pure event-study regression. The actual **operational** target definition — §3.2.2, Table 4 — defines:

- **Y1**: the *raw* continuously-compounded log return, $\ln(P_{t_0} / P_{t_0-1})$. No market model, no α/β. `build_targets()` implements this correctly.
- **Y2**: raw trading volume during the event window, *relative to the 30-day pre-event moving average*. Implemented correctly.
- **Y3**: consecutive trading days for ASPI to return to its pre-event baseline, capped at 90.

**Y3 deviates from that specification, and this audit found it.** `build_targets()` searches a 91-**trading-row** window but returns `(recovery_date - event_date).days` — a **calendar**-day difference (`feature_eng.py:123`). The 90-day cap therefore bites at roughly 62 trading days, not 90, and an event recovering after ~100 calendar days is recorded as 90, indistinguishable from "never recovered". An earlier version of this cell asserted the implementation "was correct all along"; that claim was wrong and is corrected here.

**A structural property of Y3 that must be stated before any Y3 result is read:** the recovery search window includes the event day itself and the baseline is $P_{t_0-1}$, so whenever $P_{t_0} \geq P_{t_0-1}$ — that is, whenever **Y1 ≥ 0** — the event day itself satisfies the recovery condition and **Y3 = 0 exactly**. Y3 = 0 and Y1 ≥ 0 are therefore the same event, by construction. This is why Y3's median is 0 (over half of all events), and it means Y3 is a zero-inflated, right-censored variable whose variance is substantially explained by the sign of Y1 alone. Any Y3 model result — including the positive R-squared reported in §10 — is partly a sign-classification result in regression form, and should be read that way.

In [7]:
targets = fe.build_targets(market_clean, in_scope)
print(f"Targets built for {targets.shape[0]} in-scope events")
targets.describe()


Targets built for 74 in-scope events


,event_date,Y1_ASPI_5D_Forward_LogReturn_Pct,Y1_horizon_end_date,Y2_label_end_date,Y3_label_end_date,Y2_abnormal_volume,Y3_recovery_days,Y1_EventWindow_0_10_LogReturn_Pct,Y1_EventWindow_0_10_horizon_end_date
count,74,74.000000,74,74,74,61.000000,74.000000,74.000000,74
mean,2015-02-20 09:24:19.459459584,0.432799,2015-02-28 10:42:09.729729792,2015-02-21 03:14:35.675675648,2015-03-17 14:35:40.540540416,-0.129247,16.554054,0.748843,2015-03-08 05:30:48.648648704
min,2000-09-18 00:00:00,-7.203287,2000-09-25 00:00:00,2000-09-18 00:00:00,2000-09-20 00:00:00,-0.869890,0.000000,-11.980773,2000-10-02 00:00:00
25%,2009-11-26 18:00:00,-1.071805,2009-12-05 06:00:00,2009-11-28 06:00:00,2009-12-05 06:00:00,-0.530635,0.000000,-2.052479,2009-12-13 06:00:00
50%,2015-10-15 12:00:00,0.119692,2015-10-23 00:00:00,2015-10-16 00:00:00,2015-11-17 12:00:00,-0.345942,4.500000,0.349066,2015-10-31 00:00:00
75%,2020-12-24 12:00:00,1.775149,2021-01-02 18:00:00,2020-12-26 18:00:00,2020-12-26 18:00:00,0.161990,16.750000,3.887791,2021-01-10 12:00:00
max,2025-11-27 00:00:00,9.009358,2025-12-05 00:00:00,2025-11-27 00:00:00,2026-01-02 00:00:00,1.942153,90.000000,16.150356,2025-12-12 00:00:00
std,NaN,3.052712,NaN,NaN,NaN,0.579882,27.653998,4.982610,NaN


### 2.5.1 Inspect the constructed targets

A first look at the five target columns before anything is modelled. `NaN` here is
meaningful — it marks events whose volume was never recorded, and those events are
dropped from Y2 by the per-target mask rather than filled.

In [8]:
targets.head(10)


,event_date,Y1_ASPI_5D_Forward_LogReturn_Pct,Y1_horizon_end_date,Y2_label_end_date,Y3_label_end_date,Y2_abnormal_volume,Y3_recovery_days,Y3_censored,Y3_censor_reason,Y3_drawdown_occurred,Y1_EventWindow_0_10_LogReturn_Pct,Y1_EventWindow_0_10_horizon_end_date
0,2000-09-18,2.426049,2000-09-25,2000-09-18,2000-09-20,NaN,2.0,False,recovered,True,5.692355,2000-10-02
1,2000-11-18,-7.203287,2000-11-27,2000-11-20,2000-12-27,NaN,25.0,True,next_disaster,True,-9.432678,2000-12-04
2,2000-12-24,-0.044524,2001-01-04,2000-12-27,2001-01-05,NaN,6.0,False,recovered,True,0.996576,2001-01-12
3,2001-09-01,0.122684,2001-09-10,2001-09-03,2001-09-07,-0.088211,4.0,False,recovered,True,0.318667,2001-09-17
4,2002-12-16,2.477690,2002-12-24,2002-12-16,2002-12-18,-0.530635,2.0,False,recovered,True,4.504703,2003-01-02
5,2003-05-17,-1.106956,2003-05-26,2003-05-19,2003-05-28,-0.824716,7.0,False,recovered,True,1.129956,2003-06-02
6,2004-12-11,1.195929,2004-12-20,2004-12-13,2004-12-13,-0.471526,0.0,False,recovered,False,-2.258627,2004-12-29
7,2004-12-26,-3.086804,2005-01-04,2004-12-28,2005-01-13,-0.439913,12.0,False,recovered,True,-0.669944,2005-01-11
8,2005-11-21,-2.217702,2005-11-28,2005-11-21,2006-04-05,0.393988,90.0,True,cap_90,True,-6.261787,2005-12-05
9,2006-10-26,-0.452482,2006-11-02,2006-10-26,2006-11-03,-0.353236,6.0,False,recovered,True,2.851568,2006-11-09


## 2.6 Assembling the event-level table

Each qualifying disaster becomes one training row: its own exogenous features (damage, population, type), the market's engineered endogenous features **as of the day before the event** (never the shock day itself — enforced explicitly here via `asof_date = event_date - 1`, on top of the feature-level shift already applied in §4), and the macro/global control values as of that same date.

(Repo note: `feature_eng.build_feature_table()` exists but is designed as a running daily panel with disaster info merge_asof'd backward — useful for a different framing, not directly for this one-row-per-event table. Assembled directly here instead, to avoid a column-collision it produces when composed with a second event-level merge.)

In [9]:
events_sorted = in_scope.sort_values("event_date").copy()
events_sorted["asof_date"] = events_sorted["event_date"] - pd.Timedelta(days=1)

# --- New: disaster recency features (real event dates, no new source) ---
# days_since_last_disaster: gap to the previous in-scope event. The very first
# event has no predecessor -- filled with a fixed 10-year sentinel (3650 days)
# representing "no recent in-scope disaster", not a fabricated average.
gap_days = events_sorted["event_date"].diff().dt.days
events_sorted["days_since_last_disaster"] = gap_days.fillna(3650.0)


def _trailing_count(dates: pd.Series, window_days: int = 365) -> list[int]:
    d = dates.values
    return [int(((d > (di - np.timedelta64(window_days, "D"))) & (d < di)).sum()) for di in d]


events_sorted["disasters_trailing_365d"] = _trailing_count(events_sorted["event_date"])

snap = pd.merge_asof(
    events_sorted.sort_values("asof_date"), market_feats.sort_values("date"),
    left_on="asof_date", right_on="date", direction="backward",
)
snap = pd.merge_asof(
    snap.sort_values("asof_date"),
    macro[["date", "gdp_growth_pct", "inflation_cpi_pct", "gdp_current_usd"]].sort_values("date"),
    left_on="asof_date", right_on="date", direction="backward", suffixes=("", "_macro"),
)
snap = pd.merge_asof(
    snap.sort_values("asof_date"), sp500.sort_values("date"),
    left_on="asof_date", right_on="date", direction="backward", suffixes=("", "_gspc"),
)

# --- New: damage-to-GDP ratio -- ties disaster magnitude to the economy's
# real scale that year, which raw log-damage alone doesn't capture. Both
# financial_damage and gdp_current_usd are already in raw USD (not '000s) --
# see emdat_loader.py and macro_loader.py docstrings.
snap["damage_to_gdp"] = (snap["financial_damage"] / snap["gdp_current_usd"]).replace([np.inf, -np.inf], np.nan)

dataset = pd.merge(snap, targets, on="event_date", how="inner").sort_values("event_date").reset_index(drop=True)

# --- External data blocks A-D, specified in docs/EXTERNAL_DATA_PRE_DECLARATION.md -----
# Written and committed BEFORE any of these features was scored, per integrity
# constraint 2. Four sources, all reachability-tested, all cached offline:
#
#   A hazard      NASA POWER daily precipitation and wind, 7 district points. Measured
#                 by instrument, so unlike financial_damage it has no reporting bias and
#                 no missingness -- and it is knowable on the event day, which makes it
#                 admissible in the ex-ante Model A specification.
#   B desinventar UNDRR/UNDP national loss database, district-level physical severity.
#                 Carries NO monetary loss (valorus is populated in 0 of 130,018
#                 records), so it substitutes for the missing damage figures in physical
#                 terms only. Ends 2020-12-20; di_available flags that.
#   C fx          FRED DEXSLUS daily LKR/USD, replacing an ANNUAL macro series matched
#                 against day-0 events. All three terms use data at or before t-1, the
#                 same pre-shock boundary as the ASPI features.
#   D election    Wikidata national election dates. The 2005-11-17 presidential election
#                 falls 4 days before the 2005-11-21 event that carries the largest
#                 observed Y1 drop.
#
# Motivation, measured: financial_damage is real for 17/64 events and zero-filled for 47.
# Every column below is 100% covered on all events.
from src.data_pipeline.external_sources import (
    EXTERNAL_FEATURE_BLOCKS, build_all_external_features)

EXTERNAL_CACHE = ARTIFACT_DIR / "external"
external_feats = build_all_external_features(dataset["event_date"], EXTERNAL_CACHE)
dataset = dataset.merge(external_feats, on="event_date", how="left")

print("External feature coverage (non-null %, all events):")
for _blk, _cols in EXTERNAL_FEATURE_BLOCKS.items():
    _cov = dataset[_cols].notna().mean().mul(100).round(1)
    print(f"  {_blk:12s} {_cov.min():5.1f}-{_cov.max():5.1f}%  ({len(_cols)} features)")
print(f"  DesInventar matched: {int(dataset['di_available'].sum())}/{len(dataset)} events "
      f"(the database ends 2020-12-20)")
print(f"  events within +/-5 days of a national election: "
      f"{int(dataset['election_within_5d'].sum())}")
print()

# --- New: one limited interaction term (Flood only -- 51/74 of the
# qualifying set, the only type with enough rows for an interaction to mean
# anything at this N; all 5 type interactions were considered and rejected
# as too many features for N~64). ---
# np.where, not a plain product: NaN * 0.0 == NaN in float arithmetic, which would make
# every non-Flood event's interaction term spuriously NaN whenever damage is missing
# (methodology-audit finding #14 fix surfaced this -- financial_damage/log_financial_damage
# can now be real NaN). Only a FLOOD event can have a genuinely missing interaction
# value; every non-Flood event is definitionally 0, never "missing".
dataset["log_damage_x_flood"] = np.where(
    dataset["disaster_Flood"] == 1, dataset["log_financial_damage"], 0.0)

# Missingness flags for the feature-level `.fillna(0.0)` below (methodology-audit
# finding #14): without these, a zero-filled volume ratio / GARCH forecast / macro rate
# is indistinguishable from a genuinely-observed zero. One flag per group of columns
# that are always missing TOGETHER (same underlying data gap -- volume absent for 2000
# and post-2023, GARCH needing enough history to fit, an annual macro series not yet
# published for the most recent events), not one per column, to avoid manufacturing
# near-duplicate near-constant features. `deaths_available`/`homeless_available`/
# `mag_area_available`/`mag_wind_available`/`di_available`/`financial_damage_observed`
# already cover the EM-DAT/DesInventar severity columns (emdat_loader.py); this closes
# the remaining gap.
_vol_ratio_cols = [c for c in ("vol_ratio_1_30", "vol_ratio_5_30", "vol_ratio_10_30",
                                "vol_cv_30", "log_vol_change_1") if c in dataset.columns]
if _vol_ratio_cols:
    dataset["volume_features_available"] = dataset[_vol_ratio_cols].notna().all(axis=1).astype(float)
if "garch_cond_vol" in dataset.columns:
    dataset["garch_cond_vol_available"] = dataset["garch_cond_vol"].notna().astype(float)
if "gdp_growth_pct" in dataset.columns:
    dataset["macro_available"] = (dataset["gdp_growth_pct"].notna()
                                  & dataset["inflation_cpi_pct"].notna()).astype(float)

# TARGET_COLS is imported from _shared.py so the list lives in ONE place, and the
# exclusions are DERIVED from it rather than repeating the names. A target that is not
# excluded silently becomes a feature, and a forward-looking target used as a feature is
# a catastrophic leak. Not hypothetical: adding Y1_EventWindow_0_5_LogReturn_Pct / Y1_EventWindow_0_10_LogReturn_Pct put the 5- and
# 10-trading-day-ahead cumulative returns straight into FEATURE_COLS on the first run,
# which would have let every model read the future it was being asked to predict. The
# assertion below turns any repeat into a hard failure.
TARGET_COLS = list(TARGET_COLS)   # the list imported from _shared.py

EXCLUDE_COLS = {
    "event_date", "asof_date", "date", "date_macro", "date_gspc", "disaster_type", "disaster_group",
    "disaster_subgroup", "damage_source", "dis_no", "event_name", "aspi_close", "trading_volume",
    "price_source", "volume_source", "event_date_precision",
    "gdp_current_usd",  # used only to derive damage_to_gdp, not a feature itself
    # Y3 competing-risk censoring detail (methodology-audit finding #7). Y3_censored is
    # bool, which pandas' is_numeric_dtype treats as numeric -- excluded explicitly so it
    # is never auto-admitted as a feature (Y3_censor_reason is a string and is already
    # excluded by the numeric-dtype filter below, listed here anyway for visibility).
    "Y3_censored", "Y3_censor_reason",
    # Y3 adverse-response gate detail (methodology-audit finding #11) -- computed from
    # POST-event prices, same reason as Y3_censored above: diagnostic only, never a
    # feature.
    "Y3_drawdown_occurred",
}
EXCLUDE_COLS |= set(TARGET_COLS)   # every target, present and future
# Columns the stationarity tests in \u00a74 rejected. These are the raw ASPI price
# levels (sma_*/ema_*): retained in the market table because the price-relative
# ratios are computed FROM them, but never fed to a model. Their stationary
# counterparts (price_to_sma_*, price_to_ema_*) carry the same momentum signal.
# Driven by the test result, not by held-out performance.
EXCLUDE_COLS |= set(NON_STATIONARY_COLS)
FEATURE_COLS = [c for c in dataset.columns if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(dataset[c])]
# The mutually-exclusive one-hot block, declared explicitly rather than sniffed by
# prefix: SMOGN must copy these verbatim instead of blending them, and must only
# pair events that share a type.
TYPE_COLS = [c for c in FEATURE_COLS if c.startswith("disaster_")]

_leaked = sorted(set(FEATURE_COLS) & set(TARGET_COLS))
assert not _leaked, f"targets leaked into FEATURE_COLS: {_leaked}"

# What the damage features actually measure. EM-DAT records no damage estimate for
# most Sri Lankan events, and the loader zero-fills those -- so these columns behave
# substantially as an indicator of EM-DAT REPORTING COVERAGE, which is itself
# correlated with event severity and recency, rather than as a clean severity
# measure. Printed for the modelled subset, since the raw-file figure is the wrong
# denominator.
if "damage_source" in dataset.columns:
    print("Damage provenance across the modelled events:")
    print(dataset["damage_source"].value_counts().to_string())
    _has_damage = (dataset["financial_damage"] > 0).sum()
    print(f"  events with a non-zero damage figure: {_has_damage}/{len(dataset)}")
    if "disaster_Flood" in dataset.columns:
        _flood_dmg = int(((dataset["financial_damage"] > 0) & (dataset["disaster_Flood"] == 1)).sum())
        print(f"  of which floods, the support of the log_damage_x_flood term: {_flood_dmg}")
    print()

# Y3 window contamination -- FIXED (methodology-audit finding #7, 2026-09-16):
# `feature_eng.build_targets` now caps each event's recovery search at
# min(90, days_to_next_qualifying_disaster) directly, using TRADING-day positions (not
# the calendar-day proxy below), and records the outcome in `Y3_censored`/
# `Y3_censor_reason`. `preprocessor.truncate_overlapping_windows()` remains unused --
# superseded by the in-line fix rather than wired in -- and is not the mechanism this
# count is now describing. Kept as a calendar-day sanity count, not a remaining gap:
_gaps = dataset["event_date"].diff().dt.days.shift(-1)
print(f"Events with another qualifying disaster within 90 calendar days (informational; "
      f"Y3 itself is now capped on the actual trading-day gap, see Y3_censor_reason): "
      f"{int((_gaps <= 90).sum())}/{len(dataset)}")
print(f"Y3_censor_reason breakdown: "
      f"{dataset['Y3_censor_reason'].value_counts().to_dict() if 'Y3_censor_reason' in dataset.columns else 'n/a'}")
print()

X = dataset[FEATURE_COLS].fillna(0.0)
# Targets keep their NaNs. A missing target is a missing OBSERVATION, not a zero:
# Y2 is absent for 3 events whose source workbook failed to parse, and filling them
# with 0.0 would assert "volume exactly at its 30-day baseline" -- a fabricated
# measurement, in a study whose central claim is that it uses only real data.
# Section 9 drops these rows per target instead.
y = dataset[TARGET_COLS].copy()
missing_targets = y.isna().sum()
if missing_targets.any():
    print("Missing target values (rows dropped per target, never imputed):")
    for _t, _n in missing_targets[missing_targets > 0].items():
        _dates = dataset.loc[y[_t].isna(), "event_date"].dt.date.tolist()
        print(f"  {_t}: {_n} missing -> effective N={len(y) - _n}; events {_dates}")

print(f"Final training table: {X.shape[0]} events x {X.shape[1]} features")
print(f"Feature columns: {FEATURE_COLS}")
dataset[["event_date", "disaster_type"] + TARGET_COLS].head(10)


External feature coverage (non-null %, all events):
  hazard       100.0-100.0%  (6 features)
  desinventar  100.0-100.0%  (7 features)
  fx           100.0-100.0%  (3 features)
  election     100.0-100.0%  (2 features)
  DesInventar matched: 52/74 events (the database ends 2020-12-20)
  events within +/-5 days of a national election: 1

Damage provenance across the modelled events:
damage_source
missing_median_imputed    56
emdat_cpi_adjusted        18
  events with a non-zero damage figure: 18/74
  of which floods, the support of the log_damage_x_flood term: 12

Events with another qualifying disaster within 90 calendar days (informational; Y3 itself is now capped on the actual trading-day gap, see Y3_censor_reason): 35/74
Y3_censor_reason breakdown: {'recovered': 57, 'next_disaster': 9, 'cap_90': 8}

Missing target values (rows dropped per target, never imputed):
  Y2_abnormal_volume: 13 missing -> effective N=61; events [datetime.date(2000, 9, 18), datetime.date(2000, 11, 18), date

,event_date,disaster_type,Y1_ASPI_5D_Forward_LogReturn_Pct,Y2_abnormal_volume,Y3_recovery_days,Y1_EventWindow_0_10_LogReturn_Pct
0,2000-09-18,Flood,2.426049,NaN,2.0,5.692355
1,2000-11-18,Flood,-7.203287,NaN,25.0,-9.432678
2,2000-12-24,Storm,-0.044524,NaN,6.0,0.996576
3,2001-09-01,Drought,0.122684,-0.088211,4.0,0.318667
4,2002-12-16,Flood,2.477690,-0.530635,2.0,4.504703
5,2003-05-17,Flood,-1.106956,-0.824716,7.0,1.129956
6,2004-12-11,Flood,1.195929,-0.471526,0.0,-2.258627
7,2004-12-26,Other,-3.086804,-0.439913,12.0,-0.669944
8,2005-11-21,Flood,-2.217702,0.393988,90.0,-6.261787
9,2006-10-26,Flood,-0.452482,-0.353236,6.0,2.851568


## 2.7 Chronological partitioning and time-aware SMOGN

**Deviation flagged and resolved -- internal conflict between the project's own two prior documents:** the earlier *proposal* (\u00a77.5) specifies stratified 5-fold CV inside an 80% training block. The later, more developed *full thesis* (\u00a73.7.1) explicitly **bans** k-fold anywhere: *"Standard k-fold cross-validation is banned because it randomly shuffles the temporal sequence, injecting future knowledge into the training set (look-ahead bias)"* (citing Saberironaghi et al., 2025). The thesis version is adopted here as the governing methodology -- chronological walk-forward only, no k-fold, anywhere.

**A second, related leakage bug found and fixed during this review, not just the k-fold one:** the first working version of this notebook ran SMOGN **once, globally**, before cutting walk-forward folds -- so the synthetic rows it appends could end up inside a fold's *test* split, meaning that fold's "held-out" score was partly measuring the model against its own fabricated interpolated points, not real disasters. This is the exact same category of mistake the thesis criticizes k-fold for (systematic injection of information the model shouldn't have), just applied to oversampling instead of splitting. **Fixed:** SMOGN is now applied *inside* each walk-forward fold, to that fold's training rows only -- the illustration below runs it once on the full real dataset purely to show what it does, and is not the data used for training (see \u00a79's `run_walk_forward`, which re-runs it per fold).

In [10]:
from src.training.walk_forward import generate_walk_forward_splits

TRAIN_WINDOW, TEST_WINDOW, STEP = 30, 10, 10  # small windows given N -- see \u00a712 limitations

# Real disaster count only (64) -- NOT len(X_res). Computing splits on the
# post-SMOGN length was a real bug (found during this review): synthetic rows
# sit at the tail of X_res, so a fold boundary cut on that inflated length
# could place fabricated interpolated rows inside a "held-out" test fold --
# scoring the model against its own synthetic output instead of a real event.
splits = list(generate_walk_forward_splits(len(X), train_window=TRAIN_WINDOW, test_window=TEST_WINDOW, step=STEP))
print(f"Walk-forward folds: {len(splits)} (computed on the {len(X)} real events, not the post-SMOGN count)")
for i, s in enumerate(splits):
    assert s.train_index.max() < s.test_index.min(), "chronological violation -- must never happen"
print("All folds verified strictly chronological (no look-ahead).")


Walk-forward folds: 4 (computed on the 74 real events, not the post-SMOGN count)
All folds verified strictly chronological (no look-ahead).


## 2.8 Cache the modelling table

Writes the event table, the feature specification and the fold definitions to
`artifacts/`. Every modelling stage from here on reads these rather than rebuilding
them, which is what keeps the folds identical across stages 04, 05 and 06.

In [11]:
save_frame(dataset, "dataset", f"{len(dataset)} in-scope events x {len(FEATURE_COLS)} features.")
save_frame(market_feats, "market_feats", "Engineered daily market features.")
save_frame(in_scope, "in_scope", "EM-DAT events inside the archive window.")
# Global (all-real-rows) median for the flagged columns (methodology-audit finding #14),
# saved so every "refit on ALL data" consumer -- notebook 04's final SHAP refit,
# scripts/train_final_models.py, and src/inference.py's live demo -- imputes with the
# SAME numbers instead of each recomputing its own. This is deliberately NOT used
# inside any walk-forward fold (those use median_impute_from_train on that fold's real
# training rows only); it is only for the already-in-sample "refit on everything" path.
_median_cols_present = [c for c in MEDIAN_IMPUTE_COLS if c in dataset.columns]
median_impute_values = dataset[_median_cols_present].median().fillna(0.0).to_dict()

save_json({"FEATURE_COLS": FEATURE_COLS, "TARGET_COLS": TARGET_COLS, "TYPE_COLS": TYPE_COLS,
           "NON_STATIONARY_COLS": sorted(NON_STATIONARY_COLS),
           "MEDIAN_IMPUTE_VALUES": median_impute_values}, "feature_spec")
save_object(splits, "splits", f"{len(splits)} chronological walk-forward folds (train/test/step = {TRAIN_WINDOW}/{TEST_WINDOW}/{STEP}).")


cached dataset.parquet  (74 rows x 102 cols)


cached market_feats.parquet  (6366 rows x 33 cols)


cached in_scope.parquet  (94 rows x 26 cols)
cached feature_spec.json


cached splits.pkl


WindowsPath('F:/CSE-disaster-impact-predictor/artifacts/splits.pkl')